In [3]:
! pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.14.5 environment at: d:\LLMOps\.venv-1
Checked 6 packages in 40ms


In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


Data Ingestion

In [7]:
from langchain_community.document_loaders import TextLoader


C:\Users\sneha\AppData\Local\Temp\ipykernel_6200\877266886.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [9]:
loader = TextLoader("D:/LLMOps/data/Agentic AI.txt", encoding="utf-8")
documents = loader.load()


In [10]:
documents[0].page_content[:500]  # Display the first 500 characters of the first document

'Agentic AI Systems\n\nOverview\nAgentic AI refers to AI systems that can autonomously pursue goals, make decisions, use tools, plan multi-step actions, and adapt based on feedback. Unlike traditional AI systems that respond to a single prompt, agentic systems can reason about objectives and execute workflows with minimal human intervention.\n\nCore Characteristics\n1. Goal-Oriented Behavior\n   - Operate toward a defined objective.\n   - Break large goals into smaller tasks.\n\n2. Planning\n   - Create ste'

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [13]:
text_chunks = text_splitter.split_documents(documents)

In [14]:
text_chunks[:20]

[Document(metadata={'source': 'D:/LLMOps/data/Agentic AI.txt'}, page_content='Agentic AI Systems'),
 Document(metadata={'source': 'D:/LLMOps/data/Agentic AI.txt'}, page_content='Overview'),
 Document(metadata={'source': 'D:/LLMOps/data/Agentic AI.txt'}, page_content='Agentic AI refers to AI systems that can autonomously pursue goals, make decisions, use tools, plan multi-step actions, and adapt based on feedback. Unlike traditional AI systems that respond to a'),
 Document(metadata={'source': 'D:/LLMOps/data/Agentic AI.txt'}, page_content='that respond to a single prompt, agentic systems can reason about objectives and execute workflows with minimal human intervention.'),
 Document(metadata={'source': 'D:/LLMOps/data/Agentic AI.txt'}, page_content='Core Characteristics\n1. Goal-Oriented Behavior\n   - Operate toward a defined objective.\n   - Break large goals into smaller tasks.'),
 Document(metadata={'source': 'D:/LLMOps/data/Agentic AI.txt'}, page_content='2. Planning\n   - Create s

In [15]:
! uv pip install faiss-cpu

Using Python 3.14.5 environment at: d:\LLMOps\.venv-1
Checked 1 package in 9ms


In [16]:
! uv pip install langchain-openai

Using Python 3.14.5 environment at: d:\LLMOps\.venv-1
Checked 1 package in 22ms


In [37]:
from langchain_community.vectorstores import FAISS

In [22]:
! uv pip install sentence-transformers

Using Python 3.14.5 environment at: d:\LLMOps\.venv-1
Resolved 41 packages in 1.00s
Prepared 1 package in 566ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 792ms
 + sentence-transformers==5.6.0


In [38]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6096.63it/s]


In [39]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    text_chunks,
    embeddings
)

In [ ]:
#Perform similarity search (semantic search) to retrieve relevant documents based on a query
query = "What is Agentic AI?"
docs = vectorstore.similarity_search(query,k=4)
#Display the retrieved documents
for i, doc in enumerate(docs):
    print(f"Document {i+1}:\n{doc.page_content}\n")

Document 1:
Agentic AI refers to AI systems that can autonomously pursue goals, make decisions, use tools, plan multi-step actions, and adapt based on feedback. Unlike traditional AI systems that respond to a

Document 2:
Agentic AI Systems

Document 3:
Agentic AI systems extend traditional LLMs by combining reasoning, planning, memory, and tool usage to achieve goals autonomously. They form the foundation of next-generation AI assistants and

Document 4:
that respond to a single prompt, agentic systems can reason about objectives and execute workflows with minimal human intervention.



In [29]:
from langchain_core.prompts import ChatPromptTemplate
template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [30]:
prompt = ChatPromptTemplate.from_template(template)

In [31]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [33]:
#Creating String output parser
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [34]:
!uv pip install langchain-ollama ollama

Using Python 3.14.5 environment at: d:\LLMOps\.venv-1
Resolved 30 packages in 576ms
Prepared 2 packages in 120ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 51ms
 + langchain-ollama==1.1.0
 + ollama==0.6.2


In [35]:
#Basically, the output parser is used to parse the output from the language model into a specific format. In this case, we are using a string output parser, which means that we want the output to be a simple string. This is useful for question-answering tasks where we want a concise answer without any additional formatting or structure.
from langchain_ollama import ChatOllama

llm_model = ChatOllama(
    model="llama3"
)

In [42]:
from langchain_core.runnables import RunnablePassthrough
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

In [43]:
rag_chain = (
  {"context":retriever, "question":RunnablePassthrough()}
  | prompt
  | llm_model
  | output_parser
)

In [45]:
rag_chain.invoke("What is Agentic AI and where is it used?")

'Based on the provided context, Agentic AI refers to AI systems that can autonomously pursue goals, make decisions, use tools, plan multi-step actions, and adapt based on feedback.\n\nAgentic AI systems are used in applications where autonomous decision-making and goal-directed behavior are required.'